## Step 1: Select and Understand a Dataset
- **Dataset Name:** House Price Prediction Dataset (`House Price Prediction Dataset.csv`)
- **Domain:** Real Estate Valuation & Property Price Forecasting
- **Source:** Kaggle / Open Academic Real Estate Repository
- **Description:** 2,000 house property sales records containing physical attributes (`Area`, `Bedrooms`, `Bathrooms`, `Floors`, `YearBuilt`, `Location`, `Condition`, `Garage`, and selling `Price`).
- **Target Variable ($y$):** `Price` (Continuous numerical selling price in dollars).
- **Input Feature ($X$):** `Area` (Continuous numerical living area in square feet).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Configure visualization style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (7, 5)

print("✅ Libraries imported successfully.")

## Step 2: Load and Inspect the Data
We load `House Price Prediction Dataset.csv` and inspect its dimensions, headers, data types, missing values, and duplicate records.

In [ ]:
# Load dataset
house = pd.read_csv("House Price Prediction Dataset.csv")

print(f"Dataset Dimensions: {house.shape[0]} rows × {house.shape[1]} columns")

print("\n--- Sample from Beginning (head) ---")
display(house.head())

print("\n--- Sample from End (tail) ---")
display(house.tail())

In [ ]:
print("Column Names:")
print(house.columns.tolist())

print("\nData Types:")
display(house.dtypes.to_frame("Data Type"))

print("\nSummary Statistics of Numerical Columns:")
display(house.describe().T)

print("\nMissing Values Count:")
print(house.isnull().sum())

print(f"\nExact Duplicate Records: {house.duplicated().sum()}")

## Step 3: Select Input Features and Target
- **Input Feature ($X$):** `Area` (Living area in square feet).
- **Target Variable ($y$):** `Price` (Property selling price).
- **Relevance:** Area is the fundamental driver of property valuation because greater square footage represents more usable living space, higher material investment, and larger land footprint.
- **Exclusion of Identifier:** The `Id` column is an arbitrary sequence number and is strictly excluded from predictive features.

In [ ]:
# Select input feature matrix (X) and continuous target vector (y)
X = house[["Area"]]
y = house["Price"]

print("Input Feature (X) shape:", X.shape)
print("Target Variable (y) shape:", y.shape)
print("\nFirst 5 input features:")
display(X.head())

## Step 4: Visualise the Relationship
We create a scatter plot with `Area` on the horizontal axis and `Price` on the vertical axis.
> **Note on Causation:** A visible relationship in the scatter plot demonstrates strong empirical correlation; however, correlation does not by itself prove direct causation, as confounding variables (such as neighborhood location, structural condition, or age of home) also influence market value.

In [ ]:
# Scatter plot of House Area vs. Price
plt.figure(figsize=(8, 5))
plt.scatter(house["Area"], house["Price"], alpha=0.5, color="#1f77b4", edgecolors="none")
plt.title("Scatter Plot: House Area vs. Selling Price", fontsize=13, fontweight="bold")
plt.xlabel("Area (Square Feet)", fontsize=11)
plt.ylabel("Price ($)", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

## Step 5: Split the Data
We divide the dataset into an **80% Training set** and a **20% Testing set** using `train_test_split(..., test_size=0.20, random_state=42)`.
- **Purpose of Split:** Training data is used exclusively to learn the model parameters (slope and intercept). Testing data is strictly reserved to evaluate predictive performance on unseen observations.
- **Rule Followed:** The test set is completely isolated and is never used during model fitting.

In [ ]:
# Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Total Observations       : {len(house)}")
print(f"Training Observations (80%): {len(X_train)}")
print(f"Testing Observations (20%) : {len(X_test)}")

## Step 6: Train Linear Regression
We train an ordinary least squares `LinearRegression` model on the training data and extract the learned slope ($\beta_1$) and intercept ($\beta_0$).

In [ ]:
# Initialize and fit Linear Regression model
reg = LinearRegression()
reg.fit(X_train, y_train)

slope = reg.coef_[0]
intercept = reg.intercept_

print("=" * 45)
print("🎯 LEARNED LINEAR REGRESSION PARAMETERS")
print("=" * 45)
print(f"Learned Intercept (β0) : {intercept:.4f}")
print(f"Learned Slope (β1)     : {slope:.4f}")
print("=" * 45)

print(f"\nMathematical Model Equation:")
print(f"  Price = {intercept:.2f} + ({slope:.2f} * Area)")
print(f"\nMeaning of Parameters:")
print(f"  • Slope (β1 = {round(slope, 2)}): For every additional square foot of living area, the expected house price increases by approx USD {round(slope, 2)}.")
print(f"  • Intercept (β0 = {round(intercept, 2)}): Baseline theoretical price offset when Area is zero.")

## Step 7: Generate Predictions
We use the trained model to predict property values for the unseen test inputs (`X_test`). We present actual and predicted values side-by-side.

In [ ]:
# Generate test predictions
predicted = reg.predict(X_test)

# Compare actual vs predicted prices side-by-side
results = pd.DataFrame({
    "Area (Sq Ft)": X_test["Area"].values,
    "Actual Price ($)": y_test.values,
    "Predicted Price ($)": np.round(predicted, 2),
    "Residual Error ($)": np.round(y_test.values - predicted, 2)
})

print("--- Side-by-Side Comparison of Actual vs. Predicted Prices (Sample of 10 Test Records) ---")
display(results.head(10))

## Step 8: Evaluate the Model
We evaluate model predictive quality using:
- **Mean Absolute Error (MAE):** Average magnitude of errors in target units (dollars).
- **Mean Squared Error (MSE):** Average squared difference, penalizing large residual outliers.
- **Root Mean Squared Error (RMSE):** Square root of MSE, measuring standard deviation of residuals in original units (dollars).
- **R-squared ($R^2$ Score):** Proportion of variance explained compared to a naive mean-based baseline. ($R^2$ can be negative on test data if predictions perform worse than the mean).

In [ ]:
# Calculate evaluation metrics
mae = mean_absolute_error(y_test, predicted)
mse = mean_squared_error(y_test, predicted)
rmse = mse ** 0.5
r2 = r2_score(y_test, predicted)

print("=" * 50)
print("📊 REGRESSION PERFORMANCE EVALUATION METRICS")
print("=" * 50)
print(f"Mean Absolute Error (MAE)  : {round(mae, 2)}")
print(f"Mean Squared Error (MSE)   : {round(mse, 2)}")
print(f"Root Mean Squared Error    : {round(rmse, 2)}")
print(f"R-squared (R² Score)       : {round(r2, 4)} ({round(r2 * 100, 2)}% Variance Explained)")
print("=" * 50)

## Step 9: Visualise Actual versus Predicted Values
We create visual diagnostics:
1. The fitted regression line over test points (`Area` vs `Price`).
2. Actual Price vs. Predicted Price with an ideal diagonal reference line ($y = x$).

In [ ]:
# Visualization 1: Linear Regression Line Fitted Over Test Data
plt.figure(figsize=(8, 5))
plt.scatter(X_test["Area"], y_test, alpha=0.4, color="#1976d2", label="Actual Test Observations")
plt.plot(X_test["Area"], predicted, color="red", linewidth=2.5, label="Fitted Linear Regression Line")
plt.title("Fitted Regression Line: Area vs. Selling Price", fontsize=13, fontweight="bold")
plt.xlabel("Area (Square Feet)")
plt.ylabel("Price ($)")
plt.legend()
plt.tight_layout()
plt.show()

# Visualization 2: Actual vs. Predicted Scatter Plot
plt.figure(figsize=(6, 6))
plt.scatter(y_test, predicted, alpha=0.6, color="#2e7d32", edgecolors="k", label="Predictions")
low_val = min(y_test.min(), predicted.min())
high_val = max(y_test.max(), predicted.max())
plt.plot([low_val, high_val], [low_val, high_val], "--", color="red", linewidth=2, label="Perfect Prediction Line (y = x)")
plt.title("Actual Price vs. Predicted Price", fontsize=13, fontweight="bold")
plt.xlabel("Actual Price ($)")
plt.ylabel("Predicted Price ($)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

## Step 10: Predict a New Example
We choose realistic property sizes within the training distribution (e.g. 1500, 2000, 2500, and 3000 sq ft) and generate price predictions.

In [ ]:
# Predict for new house properties
sample_areas = [1500, 2000, 2500, 3000]
new_properties = pd.DataFrame({"Area": sample_areas})
new_properties["Predicted Price ($)"] = np.round(reg.predict(new_properties[["Area"]]), 2)

print("--- Predicted Selling Prices for New Unseen Properties ---")
display(new_properties)

single_area = 2000
single_pred = reg.predict(pd.DataFrame({"Area": [single_area]}))[0]
print(f"\n🎯 A property of {single_area} sq ft has an estimated price of: {round(single_pred, 2)}")